# Módulo 1 · Etapa 7 — Pipeline completo (`make_dataset`)
**TFM · HAM10000** · Responsable: **Adrián Arias** · MLflow: experimento `m1-datos`, run `m1-pipeline`, tag `fase=pipeline`

**Objetivo:** una única tubería que entregue a cualquiera de las tres redes las imágenes correctas, con la etiqueta correcta y en el formato que cada red espera. Es la última entrega del Módulo 1 al Módulo 2.

**Pasos, en este orden** (los cuatro primeros son fijos e iguales para train, validación y prueba):

| # | Paso | Decisión | Se aplica a |
|---|---|---|---|
| 1 | Leer y decodificar el JPEG | RGB, 3 canales | Todos |
| 2 | Recorte central | 600×450 → 450×450 (se quitan 75 px a cada lado) | Todos |
| 3 | Redimensionar | 450×450 → 224×224, interpolación por área | Todos |
| 4 | Caché en memoria | uint8 (≈ 1,5 GB en total) | Todos |
| 5 | Barajar y aumentar | `src/augmentation.py` (Etapa 6), semilla distinta por imagen y época | **Solo train** |
| 6 | Normalizar | Función oficial de cada red (`keras.applications`) | Todos |
| 7 | Lotes y precarga | 32 imágenes | Todos |

**Qué entrega:** `src/data.py` con `make_dataset(split, arch, training)`, el contrato de `docs/02_flujo_de_trabajo.md`, más `split_df()` (orden de las imágenes, para guardar predicciones) y `load_class_weights()`.

Entorno: **CPU** basta (la sección 6 tarda unos minutos más). Secretos: `DAGSHUB_USER`, `DAGSHUB_TOKEN` y `KAGGLE_API_TOKEN` (para las máscaras).

## 1. Preparación de la sesión

In [ ]:
AUTOR = "adrian"
OWNER, REPO = "aReinoso007", "tfm-melanoma-cnn-gradcam"
DATASET_SHA256_ESPERADO = "06aee5e603d313ba69de38263053c792a1dff030eb9c44a96ceb477ceefb99a5"
SPLIT_SHA256_ESPERADO = "98485423d758cae1ed8b7adf64c3e92d31a6afd4d48de7a26e89355238e34b38"

In [ ]:
import os, json, time, shutil, zipfile
from pathlib import Path
from google.colab import drive, userdata
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/TFM")

REPO_DIR = f"/content/{REPO}"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/{OWNER}/{REPO}.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!pip install -q -r requirements.txt kagglehub

In [ ]:
import numpy as np, pandas as pd, tensorflow as tf
from src.tracking import load_config, file_sha256
cfg = load_config("configs/base.yaml")
print("TensorFlow", tf.__version__)

# Imágenes al disco local (rápido): la tubería las lee de aquí
LOCAL = Path(cfg["paths"]["local_data_dir"])
if not LOCAL.exists():
    t0 = time.time()
    shutil.copy(cfg["paths"]["dataset_zip"], "/content/HAM10000.zip")
    with zipfile.ZipFile("/content/HAM10000.zip") as z:
        z.extractall(LOCAL)
    os.remove("/content/HAM10000.zip")
    print(f"Descomprimido en {LOCAL} ({time.time()-t0:.0f} s)")
assert file_sha256(cfg["paths"]["metadata_csv"]) == DATASET_SHA256_ESPERADO, "[!] Metadatos distintos a los de la Etapa 0"

# El split v1 ya está en main; class_weights.json se toma de Drive si el PR #2 aún no se fusionó
assert file_sha256("data/splits/split_v1.csv") == SPLIT_SHA256_ESPERADO, "[!] El split del repo no es el v1"
if not Path("data/splits/class_weights.json").exists():
    shutil.copy(DRIVE_ROOT / "results" / "m1" / "etapa5" / "class_weights.json", "data/splits/class_weights.json")
    print("class_weights.json copiado desde Drive (PR #2 pendiente)")
print("[+] Dataset, split v1 y pesos listos")

OUT = DRIVE_ROOT / "results" / "m1" / "etapa7"; OUT.mkdir(parents=True, exist_ok=True)
FIG = DRIVE_ROOT / "figures" / "m1" / "etapa7"; FIG.mkdir(parents=True, exist_ok=True)
CODE_DRIVE = DRIVE_ROOT / "code" / "m1"; CODE_DRIVE.mkdir(parents=True, exist_ok=True)

## 2. Módulos `src/augmentation.py` (Etapa 6) y `src/data.py` (esta etapa)
Las dos celdas escriben los archivos en la carpeta del repo. Irán juntos al repositorio en un PR (sección 9).

In [ ]:
%%writefile src/augmentation.py
"""Aumento de datos del Módulo 1 (Etapa 6). Solo se aplica en entrenamiento.

Entrada: imagen float32 (H, W, 3) en [0, 255], ya recortada y redimensionada
(después de la caché del pipeline). Salida: misma forma y rango.

Todas las operaciones son "stateless": dado el mismo `seed` (tensor int64 de 2 valores)
la imagen aumentada es siempre la misma. El pipeline (Etapa 7) genera un seed distinto
por imagen y época, así que cada época ve variantes nuevas y el resultado es reproducible.

Dos intensidades (aumento selectivo):
  - fuerte (todas las clases salvo nv): volteos, rotación, zoom, brillo y contraste.
  - ligero (nv): solo volteos horizontal y vertical, que no pierden ni inventan píxeles.
"""
import math
import tensorflow as tf

CLASES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]

# Parámetros por defecto = los de configs/base.yaml (data.augmentation).
# Revisión visual (315 variantes, 35 lesiones, revisor no dermatólogo): oscurecimiento leve en el 5,6 %
# sin pérdida de los rasgos de la lesión; el zoom toca la lesión en el 0,7 %. Se mantienen los valores.
CONFIG_POR_DEFECTO = {
    "apply_to": ["akiec", "bcc", "bkl", "df", "mel", "vasc"],   # intensidad fuerte
    "horizontal_flip": True,
    "vertical_flip": True,
    "rotation_factor": 0.25,    # fracción de vuelta: ±0,25 × 360° = ±90°. Con los dos volteos cubre los 360°
    "zoom_factor": 0.1,         # solo acercar: el lado visible pasa a ser 90–100 % (≈ 81–100 % del área)
    "brightness_delta": 0.1,    # brillo multiplicativo ×[0,9, 1,1]
    "contrast_factor": 0.1,     # contraste ×[0,9, 1,1] respecto a la media de cada canal
    "fill_mode": "REFLECT",     # relleno al rotar: reflejo (nunca negro: fabricaría un borde de dermatoscopio)
}

def _subseeds(seed, n):
    return tf.unstack(tf.random.experimental.stateless_split(tf.cast(seed, tf.int64), num=n))

def _voltear(img, seed, cfg):
    s1, s2 = _subseeds(seed, 2)
    if cfg["horizontal_flip"]:
        img = tf.image.stateless_random_flip_left_right(img, s1)
    if cfg["vertical_flip"]:
        img = tf.image.stateless_random_flip_up_down(img, s2)
    return img

def rotar(img, angulo, fill_mode="REFLECT"):
    """Rota `angulo` radianes alrededor del centro; las esquinas se rellenan según fill_mode."""
    h = tf.cast(tf.shape(img)[0], tf.float32)
    w = tf.cast(tf.shape(img)[1], tf.float32)
    c, s = tf.cos(angulo), tf.sin(angulo)
    cx, cy = (w - 1) / 2, (h - 1) / 2
    # Transformación inversa (salida -> entrada), formato de ImageProjectiveTransformV3
    t = tf.stack([c, -s, cx - c * cx + s * cy, s, c, cy - s * cx - c * cy, 0.0, 0.0])
    out = tf.raw_ops.ImageProjectiveTransformV3(
        images=img[tf.newaxis], transforms=t[tf.newaxis], output_shape=tf.shape(img)[:2],
        fill_value=0.0, interpolation="BILINEAR", fill_mode=fill_mode)
    return out[0]

def _fuerte(img, seed, cfg):
    s_flip, s_rot, s_zoom, s_crop, s_bri, s_con = _subseeds(seed, 6)
    img = _voltear(img, s_flip, cfg)
    # Rotación continua
    if cfg["rotation_factor"] > 0:
        lim = cfg["rotation_factor"] * 2 * math.pi
        ang = tf.random.stateless_uniform([], s_rot, -lim, lim)
        img = rotar(img, ang, cfg["fill_mode"])
    # Zoom (solo acercar): recorte aleatorio de lado 90–100 % y vuelta al tamaño original
    if cfg["zoom_factor"] > 0:
        h, w = tf.shape(img)[0], tf.shape(img)[1]
        frac = tf.random.stateless_uniform([], s_zoom, 1.0 - cfg["zoom_factor"], 1.0)
        lado_h = tf.cast(tf.round(frac * tf.cast(h, tf.float32)), tf.int32)
        lado_w = tf.cast(tf.round(frac * tf.cast(w, tf.float32)), tf.int32)
        img = tf.image.stateless_random_crop(img, tf.stack([lado_h, lado_w, 3]), s_crop)
        img = tf.image.resize(img, tf.stack([h, w]), method="bilinear")
    # Brillo y contraste (no se toca tono ni saturación: el color es criterio diagnóstico)
    if cfg["brightness_delta"] > 0:
        d = cfg["brightness_delta"]
        img = img * tf.random.stateless_uniform([], s_bri, 1.0 - d, 1.0 + d)
    if cfg["contrast_factor"] > 0:
        f = cfg["contrast_factor"]
        img = tf.image.stateless_random_contrast(img, 1.0 - f, 1.0 + f, s_con)
    return tf.clip_by_value(img, 0.0, 255.0)

def _ligero(img, seed, cfg):
    return _voltear(img, seed, cfg)

def augmentar(img, etiqueta, seed, cfg=None):
    """Aumenta UNA imagen según su clase. `etiqueta` = índice de clase (orden de CLASES)."""
    cfg = {**CONFIG_POR_DEFECTO, **(cfg or {})}
    fuertes = tf.constant([CLASES.index(c) for c in cfg["apply_to"]], dtype=tf.int32)
    es_fuerte = tf.reduce_any(tf.equal(tf.cast(etiqueta, tf.int32), fuertes))
    img = tf.cast(img, tf.float32)
    return tf.cond(es_fuerte, lambda: _fuerte(img, seed, cfg), lambda: _ligero(img, seed, cfg))

def preparar(img, etiqueta, seed, training, cfg=None):
    """Punto de entrada del pipeline: aumenta solo si training=True; si no, devuelve la imagen tal cual."""
    if training:
        return augmentar(img, etiqueta, seed, cfg)
    return tf.cast(img, tf.float32)

In [ ]:
%%writefile src/data.py
"""Pipeline de datos del Módulo 1 (Etapa 7): contrato M1 -> M2.

Uso en el Módulo 2:

    from src.data import make_dataset, split_df, load_class_weights
    train_ds = make_dataset("train", "resnet50", training=True)
    val_ds   = make_dataset("val",   "resnet50", training=False)
    model.fit(train_ds, validation_data=val_ds, class_weight=load_class_weights())
    # Predicciones: val/test no se barajan; el orden es el de split_df("val") / split_df("test")

Orden de los pasos (no se debe alterar):
  1. leer y decodificar el JPEG (RGB)
  2. recorte central 600×450 -> 450×450
  3. reducción a 224×224 por área
  4. caché en memoria (uint8)            -- todo lo anterior es fijo e igual para los 3 conjuntos
  5. solo train: barajar y aumentar (src/augmentation.py), con una semilla distinta por imagen y época
  6. normalización oficial de cada arquitectura (keras.applications.<red>.preprocess_input)
  7. lotes y precarga
"""
from __future__ import annotations

import hashlib
import json
from pathlib import Path

import pandas as pd
import tensorflow as tf

from src.augmentation import CLASES, preparar

SPLIT_CSV = Path("data/splits/split_v1.csv")
CLASS_WEIGHTS_JSON = Path("data/splits/class_weights.json")
SPLIT_SHA256_V1 = "98485423d758cae1ed8b7adf64c3e92d31a6afd4d48de7a26e89355238e34b38"
IMAGES_DIR = Path("/content/data/HAM10000/images")      # = paths.images_dir de configs/base.yaml
IMG_SIZE = 224
BATCH_SIZE = 32
SEMILLA = 42
ARQUITECTURAS = ("resnet50", "densenet121", "efficientnetb0")


def _sha256(path) -> str:
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def split_df(split: str, split_csv: str | Path = SPLIT_CSV) -> pd.DataFrame:
    """Filas del split oficial (verificado por su huella), en el orden en que las entrega make_dataset."""
    assert _sha256(split_csv) == SPLIT_SHA256_V1, "El CSV del split no es el split v1 publicado"
    df = pd.read_csv(split_csv)
    assert split in ("train", "val", "test"), split
    df = df[df.split == split].sort_values("image_id").reset_index(drop=True)
    df["label"] = df.dx.map({c: i for i, c in enumerate(CLASES)}).astype("int32")
    return df


def load_class_weights(version: str = "sqrt", path: str | Path = CLASS_WEIGHTS_JSON) -> dict[int, float]:
    """Pesos por clase para model.fit(class_weight=...), calculados solo con train del split v1."""
    d = json.loads(Path(path).read_text())
    assert d["split_sha256"] == SPLIT_SHA256_V1, "class_weights.json no corresponde al split v1"
    return {i: float(d[version][c]) for i, c in enumerate(d["clases"])}


def _cargar(ruta):
    """Pasos 1–3: JPEG -> recorte central cuadrado -> 224×224 por área -> uint8."""
    img = tf.io.decode_jpeg(tf.io.read_file(ruta), channels=3)
    h, w = tf.shape(img)[0], tf.shape(img)[1]
    lado = tf.minimum(h, w)
    img = tf.image.crop_to_bounding_box(img, (h - lado) // 2, (w - lado) // 2, lado, lado)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE], method="area")
    return tf.cast(tf.round(tf.clip_by_value(img, 0, 255)), tf.uint8)


def normalizar(x, arch: str):
    """Paso 6: preprocesado OFICIAL de cada red (entrada float32 RGB en [0, 255])."""
    apps = tf.keras.applications
    f = {"resnet50": apps.resnet50.preprocess_input,          # BGR y resta de la media de ImageNet
         "densenet121": apps.densenet.preprocess_input,       # [0, 1], media y desviación de ImageNet
         "efficientnetb0": apps.efficientnet.preprocess_input  # sin cambios: el modelo normaliza dentro
         }[arch]
    return f(tf.cast(x, tf.float32))


def make_dataset(split: str, arch: str, training: bool, batch_size: int = BATCH_SIZE,
                 seed: int = SEMILLA, images_dir: str | Path = IMAGES_DIR,
                 split_csv: str | Path = SPLIT_CSV, aug_cfg: dict | None = None,
                 one_hot: bool = True, cache: bool = True) -> tf.data.Dataset:
    """Devuelve (imagen normalizada para `arch`, etiqueta) en lotes.

    training=True: baraja en cada época y aplica el aumento selectivo. Solo para split="train".
    training=False: sin barajar ni aumentar; el orden es el de split_df(split).
    """
    assert arch in ARQUITECTURAS, arch
    assert not (training and split != "train"), "El aumento solo se aplica al conjunto de entrenamiento"
    AUTO = tf.data.AUTOTUNE
    df = split_df(split, split_csv)
    rutas = [str(Path(images_dir) / f"{i}.jpg") for i in df.image_id]

    ds = tf.data.Dataset.from_tensor_slices((rutas, df.label.values))
    ds = ds.map(lambda r, y: (_cargar(r), y), num_parallel_calls=AUTO)
    if cache:
        ds = ds.cache()                                        # paso 4: antes del aumento
    if training:
        ds = ds.shuffle(len(df), seed=seed, reshuffle_each_iteration=True)
        semillas = tf.data.Dataset.random(seed=seed, rerandomize_each_iteration=True).batch(2)
        ds = tf.data.Dataset.zip((ds, semillas))
        ds = ds.map(lambda xy, s: (preparar(xy[0], xy[1], s, True, aug_cfg), xy[1]), num_parallel_calls=AUTO)
    else:
        ds = ds.map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTO)
    k = len(CLASES)
    ds = ds.map(lambda x, y: (normalizar(x, arch), tf.one_hot(y, k) if one_hot else y), num_parallel_calls=AUTO)
    return ds.batch(batch_size).prefetch(AUTO)


def desnormalizar(x, arch: str):
    """Inverso aproximado de normalizar(), solo para la comprobación visual (devuelve RGB en [0, 255])."""
    x = tf.cast(x, tf.float32)
    if arch == "resnet50":
        x = (x + tf.constant([103.939, 116.779, 123.68]))[..., ::-1]
    elif arch == "densenet121":
        x = (x * tf.constant([0.229, 0.224, 0.225]) + tf.constant([0.485, 0.456, 0.406])) * 255.0
    return tf.clip_by_value(x, 0, 255)

In [ ]:
import importlib, src.augmentation, src.data
importlib.reload(src.augmentation); importlib.reload(src.data)
import src.data as D
for f in ["augmentation.py", "data.py"]:
    shutil.copy(f"src/{f}", CODE_DRIVE / f)
assert str(D.IMAGES_DIR) == cfg["paths"]["images_dir"] and D.IMG_SIZE == cfg["data"]["img_size"] \
       and D.BATCH_SIZE == cfg["training"]["batch_size"] and D.SEMILLA == cfg["data"]["split_seed"]
print("[+] Módulos cargados y coherentes con base.yaml (rutas, tamaño 224, lote 32, semilla 42)")

## 3. Justificación del recorte central con las máscaras de segmentación
Se descargan las máscaras públicas de HAM10000 (copia en Kaggle: `surajghuwalewala/ham1000-segmentation-and-classification`, ≈ 3 GB, solo en el disco de la sesión, no en Drive) y, **solo para las imágenes de train**, se mide:
- qué porcentaje de lesiones **toca** las bandas de 75 px que quita el recorte,
- qué porcentaje pierde **más del 10 %** de su área,
- cuántas lesiones ya estaban **cortadas por la propia foto** (tocan el borde de la imagen original).

La alternativa, aplastar a cuadrado, no pierde nada pero estrecha la imagen un 25 % en horizontal: un lunar redondo pasa a ser un óvalo, y eso altera la asimetría (criterio A).

Si la descarga falla, la sección se salta y el notebook sigue; el recorte queda justificado solo por el argumento geométrico.

In [ ]:
# --- Análisis de máscaras: ¿cuánto de la lesión cae en las bandas que quita el recorte central? ---
import re
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

def indexar_mascaras(raiz):
    """{image_id: ruta} de todas las máscaras PNG que haya bajo `raiz`."""
    out = {}
    for p in Path(raiz).rglob("*.png"):
        m = re.search(r"(ISIC_\d{7})", p.name)
        if m:
            out.setdefault(m.group(1), str(p))
    return out

def medir_mascara(ruta, ancho=600, alto=450):
    """Fracción de la lesión que queda fuera del recorte central y si la lesión toca el borde de la foto."""
    with Image.open(ruta) as im:
        m = im.convert("L")
        if m.size != (ancho, alto):
            m = m.resize((ancho, alto), Image.NEAREST)
        m = np.asarray(m) > 127
    total = int(m.sum())
    margen = (ancho - alto) // 2                     # 75 px a cada lado
    fuera = int(m[:, :margen].sum() + m[:, ancho - margen:].sum())
    borde_foto = bool(m[0].any() or m[-1].any() or m[:, 0].any() or m[:, -1].any())
    return dict(area_px=total, fuera_px=fuera, pct_fuera=100 * fuera / total if total else np.nan,
                toca_banda=fuera > 0, toca_borde_foto=borde_foto)

def resumen_recorte(med):
    g = med.groupby("dx")
    r = pd.DataFrame({
        "n": g.size(),
        "pct_toca_banda": 100 * g.toca_banda.mean(),
        "pct_pierde_mas_10": 100 * g.pct_fuera.apply(lambda s: (s > 10).mean()),
        "mediana_pct_fuera_si_toca": g.apply(lambda d: d.loc[d.toca_banda, "pct_fuera"].median()),
        "pct_ya_cortada_por_la_foto": 100 * g.toca_borde_foto.mean(),
    })
    tot = pd.DataFrame({"n": [len(med)], "pct_toca_banda": [100 * med.toca_banda.mean()],
                        "pct_pierde_mas_10": [100 * (med.pct_fuera > 10).mean()],
                        "mediana_pct_fuera_si_toca": [med.loc[med.toca_banda, "pct_fuera"].median()],
                        "pct_ya_cortada_por_la_foto": [100 * med.toca_borde_foto.mean()]}, index=["total"])
    return pd.concat([r, tot]).round(1)

In [ ]:
med = None
try:
    import kagglehub
    tok = userdata.get("KAGGLE_API_TOKEN")
    if tok: os.environ["KAGGLE_API_TOKEN"] = tok
    t0 = time.time()
    raiz = kagglehub.dataset_download("surajghuwalewala/ham1000-segmentation-and-classification")
    masks = indexar_mascaras(raiz)
    print(f"{len(masks)} máscaras encontradas ({time.time()-t0:.0f} s)")
    tr = D.split_df("train")
    tr = tr[tr.image_id.isin(masks)]
    print("Imágenes de train con máscara:", len(tr), "de", len(D.split_df("train")))
    from concurrent.futures import ThreadPoolExecutor
    with ThreadPoolExecutor(8) as ex:
        filas = list(ex.map(lambda r: dict(image_id=r[0], dx=r[1], **medir_mascara(masks[r[0]])),
                            tr[["image_id", "dx"]].itertuples(index=False)))
    med = pd.DataFrame(filas).dropna(subset=["pct_fuera"])
    med.to_csv(OUT / "mascaras_recorte_train.csv", index=False)
    res_rec = resumen_recorte(med); res_rec.to_csv(OUT / "resumen_recorte.csv"); display(res_rec)
    print("Columnas: % que toca las bandas · % que pierde más del 10 % de su área · mediana de área perdida "
          "entre las que tocan · % que ya llega al borde de la foto original")
except Exception as e:
    print("[!] No se pudieron analizar las máscaras:", type(e).__name__, e)
    print("La sección se omite; el recorte se justifica solo con el argumento geométrico.")

## 4. Comprobación de etiquetas
Un recorrido completo de cada conjunto: el recuento de clases debe ser idéntico a la tabla del split. Así se detecta cualquier imagen perdida o etiqueta cambiada.

In [ ]:
from src.augmentation import CLASES
for split, training in [("train", True), ("val", False), ("test", False)]:
    t0 = time.time()
    ds = D.make_dataset(split, "efficientnetb0", training)
    cuenta = np.zeros(7, int)
    for _, y in ds:
        cuenta += np.bincount(y.numpy().argmax(1), minlength=7)
    esperado = D.split_df(split).label.value_counts().reindex(range(7), fill_value=0).values
    assert np.array_equal(cuenta, esperado), f"[!] {split}: {cuenta} vs {esperado}"
    print(f"[+] {split:5s}: {dict(zip(CLASES, cuenta.tolist()))}  ({time.time()-t0:.0f} s)")
# Para el conjunto de prueba solo se cuentan etiquetas: no se mira ninguna imagen.

## 5. Determinismo, orden y protección

In [ ]:
v1 = np.concatenate([x.numpy() for x, _ in D.make_dataset("val", "resnet50", False)])
v2 = np.concatenate([x.numpy() for x, _ in D.make_dataset("val", "resnet50", False)])
assert np.array_equal(v1, v2); print("[+] Validación: mismo tensor en dos recorridos (sin aumento ni barajado)")

lab = np.concatenate([y.numpy().argmax(1) for _, y in D.make_dataset("val", "resnet50", False)])
assert np.array_equal(lab, D.split_df("val").label.values)
print("[+] Validación: el orden coincide con split_df('val') (para guardar predicciones con su image_id)")

ds = D.make_dataset("train", "resnet50", True)
e1 = np.concatenate([y.numpy().argmax(1) for _, y in ds]); e2 = np.concatenate([y.numpy().argmax(1) for _, y in ds])
assert not np.array_equal(e1, e2); print("[+] Train: el orden cambia en cada época")
xa = next(iter(D.make_dataset("train", "resnet50", True)))[0].numpy()
xb = next(iter(D.make_dataset("train", "resnet50", True)))[0].numpy()
assert np.allclose(xa, xb); print("[+] Train: reproducible (mismo primer lote en dos ejecuciones con la misma semilla)")

try:
    D.make_dataset("val", "resnet50", True); print("[!] No se protegió")
except AssertionError:
    print("[+] Protección: no se puede activar el aumento fuera de train")
cw = D.load_class_weights(); print("[+] Pesos de clase:", {k: round(v, 3) for k, v in cw.items()})

> El aviso *\"The calling iterator did not fully read the dataset being cached\"* que puede aparecer al sacar un solo lote es normal: la caché se completa en el primer recorrido entero.

## 6. Comprobación visual para las tres redes
Se saca un lote de **train** y otro de **validación**, se deshace la normalización y se muestran las imágenes con su etiqueta. Las tres filas de cada red deben verse **iguales en colores** (si alguna sale azulada o muy oscura, la normalización está mal) y las de train, con las variaciones del aumento.

In [ ]:
import matplotlib.pyplot as plt
N = 8
for split, training in [("train", True), ("val", False)]:
    fig, axes = plt.subplots(3, N, figsize=(1.6 * N, 5.2))
    for r, arch in enumerate(D.ARQUITECTURAS):
        x, y = next(iter(D.make_dataset(split, arch, training)))
        rgb = D.desnormalizar(x[:N], arch).numpy().astype("uint8")
        rango = f"[{float(tf.reduce_min(x)):.1f}, {float(tf.reduce_max(x)):.1f}]"
        for j in range(N):
            ax = axes[r, j]; ax.imshow(rgb[j]); ax.set_xticks([]); ax.set_yticks([])
            ax.set_title(CLASES[int(y[j].numpy().argmax())], fontsize=8)
        axes[r, 0].set_ylabel(f"{arch}\n{rango}", rotation=0, ha="right", va="center", fontsize=8, labelpad=8)
    fig.suptitle(f"Lote de {split} tras deshacer la normalización", fontsize=10)
    fig.tight_layout()
    fig.savefig(FIG / f"fig_lote_{split}.png", dpi=200, bbox_inches="tight", facecolor="white"); plt.show()
print("Rangos esperados: resnet50 ≈ [-124, 152] · densenet121 ≈ [-2,1, 2,6] · efficientnetb0 [0, 255]")

## 7. Tiempo de una época sin modelo
Si la lectura de datos es lenta, la GPU esperaría. La primera época incluye la lectura del disco; la segunda ya sale de la caché.

In [ ]:
tiempos = {}
ds = D.make_dataset("train", "resnet50", True)
for ep in (1, 2):
    t0 = time.time(); n = sum(int(x.shape[0]) for x, _ in ds); tiempos[ep] = time.time() - t0
    print(f"Época {ep}: {n} imágenes en {tiempos[ep]:.1f} s ({n / tiempos[ep]:.0f} img/s)")

## 8. Registro en MLflow

In [ ]:
import mlflow
from src.tracking import setup_mlflow, standard_tags
os.environ["MLFLOW_TRACKING_USERNAME"] = userdata.get("DAGSHUB_USER")
os.environ["MLFLOW_TRACKING_PASSWORD"] = userdata.get("DAGSHUB_TOKEN")
setup_mlflow(cfg); mlflow.set_experiment(cfg["mlflow"]["experiments"]["m1"])
tags = standard_tags(cfg, modulo="m1", fase="pipeline", autor=AUTOR, dataset_sha256=DATASET_SHA256_ESPERADO)
assert tags["split_sha256"] == SPLIT_SHA256_ESPERADO
with mlflow.start_run(run_name="m1-pipeline", tags=tags) as run:
    mlflow.log_params({"recorte": "central_450", "redimension": "area", "img_size": D.IMG_SIZE,
                       "batch_size": D.BATCH_SIZE, "cache": "memoria_uint8", "semilla": D.SEMILLA,
                       "normalizacion": "keras.applications.preprocess_input por red",
                       "aumento": "selectivo, solo train (src/augmentation.py)"})
    m = {"t_epoca1_s": tiempos[1], "t_epoca2_s": tiempos[2]}
    if med is not None:
        m.update({"pct_lesiones_tocan_banda": float(med.toca_banda.mean() * 100),
                  "pct_lesiones_pierden_mas_10": float((med.pct_fuera > 10).mean() * 100),
                  "n_mascaras_train": len(med)})
    mlflow.log_metrics(m)
    mlflow.log_artifact("src/data.py", artifact_path="codigo")
    mlflow.log_artifact("src/augmentation.py", artifact_path="codigo")
    mlflow.log_artifacts(str(OUT), artifact_path="etapa7")
    mlflow.log_artifacts(str(FIG), artifact_path="figuras")
print("[+] Run m1-pipeline registrado:", run.info.run_id)